In [ ]:
import gradio as gr

# Load MongoDB

In [ ]:
!pip install transformers datasets -q
!pip install pymongo -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import pandas as pd
from pymongo import MongoClient
import gradio as gr
from openai import OpenAI
from colab import userdata


url = userdata.get('MGDB_IAI')

client = MongoClient(url)

db = client["Group8"]

data1 = db["labeled_training_data"]
data2= db["漢字與詞語"]

cursor1 = data1.find()
cursor2 = data2.find()

trained_data = pd.DataFrame(list(cursor1))
rubric_data = pd.DataFrame(list(cursor2))

In [ ]:
trained_data

,_id,text_segment,calculated_level,source
0,6a12bdc0d14be1651570cf7b,這個小小的第三者，似乎一生下來就得到父母的鍾愛，在她噘著小嘴唇甜蜜睡覺的時候，在她睜開烏黑的...,2,文本
1,6a12bdc0d14be1651570cf7c,我們的臥室開始有釘鎚的響聲，鐵絲安裝起來了，一道，兩道，三道，四道，五道，六道。她的尿布像一...,2,文本
2,6a12bdc0d14be1651570cf7d,兩部車子回家的公務員生活樂趣被破壞了，但是我們卻從另一方面得到了補償。我們可以捏捏嬰兒的小手...,2,文本
3,6a12bdc0d14be1651570cf7e,我們享受她給我們的一切聲音，這聲音使我們的房間格外溫暖。我們偷看她安靜時候臉上的表情，這表情...,2,文本
4,6a12bdc0d14be1651570cf7f,小太陽不怕天上雲朵的遮掩，小太陽能透過雨絲，透過尿布的迷魂陣，透過愁苦靈魂堅硬的外殻，暖烘烘...,2,文本
...,...,...,...,...
317,6a12bdc2d14be1651570d0b8,【楊喚詩集】我是忙碌的 楊喚 我是忙碌的。 我是忙碌的。 我忙於搖醒火把， 我忙於雕塑自己；...,3,詩集
318,6a12bdc2d14be1651570d0b9,我是忙碌的。 我是忙碌的。,2,詩集
319,6a12bdc2d14be1651570d0ba,【楊喚詩集】童詩：小蜘蛛 楊喚 要黏住小蚊子討厭的尖嘴巴。 要黏住小蒼蠅亂飛的小翅膀。 蜜蜂...,2,詩集
320,6a12bdc2d14be1651570d0bb,【楊喚詩集】小時候 楊喚 小時候， 在哭聲裡長大， 讓我的日子永遠蒼白憂鬱。 從落後的鄉村走...,2,詩集


In [ ]:
rubric_data

,_id,漢字,級別
0,6a12bdb7d14be165157089a1,的,1
1,6a12bdb7d14be165157089a2,是,1
2,6a12bdb7d14be165157089a3,一,1
3,6a12bdb7d14be165157089a4,個,1
4,6a12bdb7d14be165157089a5,我,1
...,...,...,...
17877,6a12bdb7d14be1651570cf76,作物,7
17878,6a12bdb7d14be1651570cf77,座右銘,7
17879,6a12bdb7d14be1651570cf78,坐鎮,7
17880,6a12bdb7d14be1651570cf79,坐姿,7


# LLM RAG

In [ ]:
import os
import pandas as pd
from pymongo import MongoClient
from google.colab import userdata
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [ ]:
def rewrite_text_by_rag(user_text, target_level):
    """
    Retrieves grading criteria and samples from pre-loaded DataFrames,
    injects them into the prompt layout for fluent few-shot learning, and calls ChatGPT.
    """
    if not user_text.strip():
        return "Error: The input text cannot be empty!"

    # Safely convert target level to an integer (1-7)
    try:
        level_num = int(float(target_level))
    except (ValueError, TypeError):
        return f"Error: Invalid level value '{target_level}'. Please enter an integer from 1 to 7."

    try:
        print(f"Fetching reference data for Level {level_num} from DataFrames...")

        # 1. Retrieve the Rubric (Grading Criteria Mapping) from rubric_data
        matched_rubrics = rubric_data[rubric_data["級別"] == level_num]

        if not matched_rubrics.empty:
            characters_list = matched_rubrics["漢字"].dropna().astype(str).tolist()
            grading_standard = ", ".join(characters_list)
        else:
            grading_standard = None

        if not grading_standard:
            grading_standard = "No detailed character rubric vocabulary found for this level in rubric_data."
            print(f"Hint: No characters found matching Level {level_num} in rubric_data.")

        # 2. Retrieve the Trained Data (Few-Shot Generation Samples) from trained_data
        matched_examples = trained_data[trained_data["calculated_level"] == level_num].head(3)

        examples_list = []
        for idx, row in matched_examples.iterrows():
            text_content = row.get("text_segment")
            if text_content:
                examples_list.append(f"[Sample {idx+1}]\n{text_content}")

        examples_context = "\n\n".join(examples_list) if examples_list else "No trained data samples found for this level."

        # 3. Prompt Engineering (Highly Optimized for Natural Fluency & Sentence Combining)
        system_prompt = f"""You are an expert in Chinese language teaching and text difficulty adaptation.
Your task is to rewrite the user-provided "Original Text" into a version that strictly complies with "Level {level_num}" difficulty.

To help you adapt the text accurately, use the following official reference materials retrieved dynamically from the database:

[Level {level_num} - Rubric Vocabulary Guidelines]
The following characters are classified explicitly under Level {level_num}. Prioritize utilizing these characters and matching structural grammar styles associated with them when composing the final output:
{grading_standard}

[Level {level_num} - Trained Data Style Samples]
Analyze the following text segments manually graded at Level {level_num}. Learn how sentences are structured, mimic their tone, vocabulary density, sentence complexity, and average length:
{examples_context}

[Fluency and Sentence Combining Instructions]
- Create a holistic, highly cohesive paragraph with a natural, human-like flow.
- DO NOT generate short, chopped-up, child-like sentences (e.g., avoid: "夜市很熱鬧。這裡有鹹酥雞。").
- You MUST actively combine related ideas into longer, compound sentences by using simple, level-appropriate transition words and patterns.
- Please utilize the following fluent connecting structures to ensure the paragraph reads naturally:
  1. Temporal clauses: "...的時候" (e.g., 走進夜市的時候...)
  2. Causal clauses: "因為...所以..."/ "...於是..."/ "...結果..." (e.g., 因為有很多好吃的，所以大家都很開心。)
  3. Progression/Addition: "...還有..." / "...而且..." / "一邊...一邊..."/  (e.g., 這裡有很好吃的炸雞，而且還有甜甜的珍珠奶茶。)
  4. Sequence: "...然後..."/ "...接著..." (e.g., 我們吃飽了，然後去散步。)

[Content Fidelity Instructions]
- Do not summarize or omit the details. Keep every food item, description, and action mentioned in the Original Text. Convert them into simple expressions instead of deleting them."""

        user_prompt = f"""Please rewrite the following [User Input Original Text] into a text matching "Level {level_num}" difficulty.

[User Input Original Text]
{user_text}

[Constraints]
1. Output ONLY the final rewritten text result as a clean, holistic paragraph.
2. The paragraph must be highly fluent and naturally connected. Combine short clauses using smooth transition patterns ("...的時候", "因為...所以...", "而且", "一邊...一邊..."). Avoid excessive commas or disjointed short statements.
3. Maintain full content fidelity without omitting any descriptive details or entities.
4. Do not include any meta-explanations, greetings, pleasantries, or prefix/suffix wrapper tags.

[Rewritten Text Output]:"""

        print(f"Invoking ChatGPT API for text adaptation...")

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7
        )

        result_text = response.choices[0].message.content.strip()
        return result_text

    except Exception as e:
        return f"An unexpected execution error occurred: {str(e)}"

# Rewrite

In [ ]:
def rewrite_text(user_text, target_level):
    """根據 MongoDB RAG 改寫文章"""
    print(f"Level {target_level} rewritting, please wait")
    try:
        level_num = int(float(target_level))
    except (ValueError, TypeError):
        return "Error: The level shall be 1-7。"

    # 取得分級單字
    matched_rubrics = rubric_data[rubric_data["級別"] == level_num]
    if not matched_rubrics.empty:
        characters_list = matched_rubrics["漢字"].dropna().astype(str).tolist()
        grading_standard = ", ".join(characters_list)
    else:
        grading_standard = "找不到對應的分級單字標準。"

    # 取得範例文本
    matched_examples = trained_data[trained_data["calculated_level"] == level_num].head(3)
    examples_list = [f"[Sample {idx+1}]\n{row.get('text_segment')}" for idx, row in matched_examples.iterrows() if row.get("text_segment")]
    examples_context = "\n\n".join(examples_list) if examples_list else "找不到該級別的訓練範例。"

    system_prompt = f"""You are an expert in Chinese language teaching. Rewrite the user's text to strictly comply with "Level {level_num}" difficulty.
[Level {level_num} - Rubric Vocabulary Guidelines]
The following characters are classified explicitly under Level {level_num}. Prioritize utilizing these characters:
{grading_standard}

[Level {level_num} - Trained Data Style Samples]
{examples_context}

[Instructions]
- Create a holistic, cohesive paragraph with a natural, human-like flow.
- Combine related ideas into longer, compound sentences using simple, level-appropriate transition words.
- Maintain full content fidelity. Keep every food item, description, and action mentioned in the Original Text."""

    user_prompt = f"""Rewrite the following text into "Level {level_num}" difficulty. Output ONLY the final rewritten paragraph.
[Original Text]
{user_text}"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: 改寫時發生錯誤: {str(e)}"


In [ ]:
def generate_reading_quiz(reading_text, target_level):
    """嚴格根據 MongoDB 的分級標準生成 5 題 JSON 格式選擇題（TOCFL 擬真題型）"""
    print("\n Quiz generating, please wait!")

    try:
        level_num = int(float(target_level))
    except (ValueError, TypeError):
        level_num = 1

    matched_rubrics = rubric_data[rubric_data["級別"] == level_num]
    if not matched_rubrics.empty:
        characters_list = matched_rubrics["漢字"].dropna().astype(str).tolist()
        grading_standard = ", ".join(characters_list)
    else:
        grading_standard = "找不到對應的分級單字標準。"

    # 這裡更新了 system_prompt，加入 TOCFL 題型的設計指示
    system_prompt = f"""You are a professional Chinese language teacher designing a TOCFL style reading test.
Based on the provided reading text, generate exactly 5 multiple-choice reading comprehension questions.

[Level {level_num} - Rubric Vocabulary Guidelines]
The following characters are explicitly classified under Level {level_num}.
{grading_standard}

[Question Style Instructions]
Mimic the official TOCFL reading comprehension style. Include these types of questions to test different reading skills:
1. Detail retrieval (細節擷取): e.g., "關於...沒提到下面哪一項？" or "根據文章，下面哪一個說法是正確的？"
2. Meaning in context (上下文推論): e.g., "文章中的「...」指的是什麼？" or "為什麼作者覺得...？"
3. Main idea (主旨大意): e.g., "這篇文章主要想告訴我們什麼？" or "這篇短文主要想傳達什麼思想？"

CRITICAL CONSTRAINT:
The questions, the 4 options, and the correct answers MUST strictly utilize ONLY the vocabulary and grammar appropriate for Level {level_num}. You must prioritize using the characters from the rubric guidelines above. Do not introduce any advanced vocabulary that exceeds the text's level.

You MUST output ONLY a valid JSON object matching this exact structure:
{{
  "questions": [
    {{
      "question": "Question text here (TOCFL style)",
      "options": ["Option A", "Option B", "Option C", "Option D"],
      "answer": "The exact text of the correct option"
    }}
  ]
}}"""

    user_prompt = f"Text to base the quiz on:\n{reading_text}"

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={ "type": "json_object" },
            temperature=0.7
        )
        quiz_data = json.loads(response.choices[0].message.content)
        return quiz_data.get("questions", [])
    except Exception as e:
        print(f"生成測驗時發生錯誤: {str(e)}")
        return []

# Interactive quiz

In [ ]:
def run_interactive_quiz(questions):
    """在終端機逐題與使用者互動，接收輸入並批改"""
    if not questions:
        print("沒有可用的題目。")
        return

    print("\n" + "="*40)
    print("Let's start！ (Please enter A,B,C,D)")
    print("="*40)

    score = 0
    feedback_log = []
    options_map = ["A", "B", "C", "D"]

    for i, q in enumerate(questions):
        print(f"\n Question{i+1}:{q['question']}")

        for j, opt in enumerate(q['options']):
            print(f"  ({options_map[j]}) {opt}")

        user_choice = ""
        while user_choice not in options_map:
            user_choice = input("Your answer(A,B,C,D): ").strip().upper()

        selected_text = q['options'][options_map.index(user_choice)]
        correct_text = q['answer']

        if selected_text == correct_text:
            print("You are right！")
            score += 1
            feedback_log.append(f"Question {i+1}: Right!")
        else:
            print(f"You are wrong, the right answer is: {correct_text}")
            feedback_log.append(f"Question {i+1}: Wrong ><  (Your answer: {selected_text} | The right answer: {correct_text})")

    print("\n" + "="*40)
    print(f"End of quiz！Your total: {score} / {len(questions)}")
    print("="*40)
    print("Records")
    for fb in feedback_log:
        print(fb)
    print("="*40)



In [ ]:
if __name__ == "__main__":
    print("=== Interactive Reading Comprehension System ===")

    # 支援輸入檔案路徑或直接貼上文字
    user_input = input("Please input the text for rewriting: \n").strip()

    # 檢查是否為檔案，若是則自動讀取
    if os.path.isfile(user_input) and user_input.endswith('.txt'):
        try:
            with open(user_input, "r", encoding="utf-8") as f:
                original_text = f.read()
            print("Success。")
        except Exception as e:
            print(f"Failed {e}")
            original_text = ""
    else:
        original_text = user_input

    target_lvl = input("Please input your target level (1-7): ").strip()

    if original_text:
        # 1. 改寫文章
        rewritten_result = rewrite_text(original_text, target_lvl)

        if not rewritten_result.startswith("Error"):
            print("\n" + "="*40)
            print(" 【Rewritten text】 ")
            print(rewritten_result)
            print("="*40)

            # 2. 生成閱讀測驗
            generated_questions = generate_reading_quiz(rewritten_result, target_lvl)

            # 3. 進行互動測驗
            run_interactive_quiz(generated_questions)
        else:
            print(rewritten_result)
    else:
        print("沒有有效的輸入文字！")

Accuracy


In [ ]:
def score_rewritten_text(rewritten_text, target_level):
    try:
        level_num = int(float(target_level))
    except (ValueError, TypeError):
        return "Error: Invalid level."

    matched_rubrics = rubric_data[rubric_data["級別"] == level_num]
    if not matched_rubrics.empty:
        characters_list = matched_rubrics["漢字"].dropna().astype(str).tolist()
        grading_standard = ", ".join(characters_list)
    else:
        grading_standard = "找不到對應的分級單字標準。"

    matched_examples = trained_data[trained_data["calculated_level"] == level_num].head(3)
    examples_list = [
        f"[Sample {idx+1}]\n{row.get('text_segment')}"
        for idx, row in matched_examples.iterrows()
        if row.get("text_segment")
    ]
    examples_context = "\n\n".join(examples_list) if examples_list else "找不到該級別的訓練範例。"

    system_prompt = f"""You are an expert Chinese language evaluator specializing in TOCFL level assessment.
Your task is to score the provided text against the official Level {level_num} grading standards.

[Level {level_num} - Official Rubric Vocabulary]
The following characters are explicitly classified under Level {level_num}:
{grading_standard}

[Level {level_num} - Reference Samples from Labeled Training Data]
These are real texts rated at Level {level_num}. Use them to calibrate your judgment:
{examples_context}

[Scoring Instructions]
Evaluate the text on the following 4 dimensions, each scored 1–5:
1. **Vocabulary Match（詞彙吻合度）**: How well does the vocabulary align with Level {level_num} rubric?
2. **Grammar Complexity（語法難度）**: Does sentence complexity match Level {level_num}?
3. **Style Similarity（風格相似度）**: Does it match the style/tone of the reference samples?
4. **Overall Level Accuracy（整體級別準確度）**: Overall, how well does this text match Level {level_num}?

Output a JSON object in exactly this format:
{{
  "vocabulary_score": <1-5>,
  "grammar_score": <1-5>,
  "style_score": <1-5>,
  "overall_score": <1-5>,
  "feedback": "<2-3 sentences explaining strengths and what could be improved>"
}}"""

    user_prompt = f"""Please score the following rewritten text for Level {level_num} compliance:

[Text to Score]
{rewritten_text}"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        score_data = json.loads(response.choices[0].message.content)
        return score_data
    except Exception as e:
        return {"error": str(e)}

#Run Gradio

In [ ]:
# ==========================================
# Gradio Interface Implementation
# ==========================================
import gradio as gr

def process_text_and_quiz(original_text, target_level):
    """
    Handles the Gradio button click:
    1. Rewrites the text.
    2. Scores the rewritten text against rubric + training data.
    3. Generates the quiz.
    4. Updates the UI components.
    """
    if not original_text.strip():
        return "Please enter some text.", "", [], gr.update(visible=False), *[gr.update(visible=False)]*5

    # 1. Rewrite Text
    rewritten_result = rewrite_text(original_text, target_level)

    if rewritten_result.startswith("Error"):
        return rewritten_result, "", [], gr.update(visible=False), *[gr.update(visible=False)]*5

    # 2. Score the rewritten text (根據資料庫分級標準與範例自我評估)
    score_result = score_rewritten_text(rewritten_result, target_level)

    if "error" not in score_result:
        avg_score = round(
            (score_result["vocabulary_score"] + score_result["grammar_score"] +
             score_result["style_score"] + score_result["overall_score"]) / 4, 1
        )
        score_display = f"""### Quality Score — Target Level {target_level}
> Evaluated against: rubric vocabulary (漢字字詞表) and labeled training data

| Dimension | Score |
|-----------|-------|
| Vocabulary Match | {score_result['vocabulary_score']} / 5 |
| Grammar Complexity | {score_result['grammar_score']} / 5 |
| Style Similarity | {score_result['style_score']} / 5 |
| Overall Level Accuracy | {score_result['overall_score']} / 5 |
| **Average** | **{avg_score} / 5** |

💬 **Feedback:** {score_result['feedback']}"""
    else:
        score_display = f"Scoring error: {score_result['error']}"

    # 3. Generate Quiz
    quiz_data = generate_reading_quiz(rewritten_result, target_level)

    # 4. Prepare UI updates for the 5 questions
    radio_updates = []
    for i in range(5):
        if i < len(quiz_data):
            q = quiz_data[i]
            radio_updates.append(gr.update(label=f"Q{i+1}: {q['question']}", choices=q['options'], visible=True, value=None))
        else:
            radio_updates.append(gr.update(visible=False))

    return rewritten_result, score_display, quiz_data, gr.update(visible=True), *radio_updates

def grade_quiz(quiz_data, q1, q2, q3, q4, q5):
    """
    Grades the user's selected answers against the stored quiz data.
    """
    if not quiz_data:
        return "No quiz data available to grade."

    user_answers = [q1, q2, q3, q4, q5]
    score = 0
    feedback = []

    for i in range(len(quiz_data)):
        correct_answer = quiz_data[i]['answer']
        user_choice = user_answers[i]

        if not user_choice:
            feedback.append(f"**Question {i+1}:** You skipped this question. (Correct answer: {correct_answer})")
            continue

        if user_choice == correct_answer:
            score += 1
            feedback.append(f"**Question {i+1}:** Correct!")
        else:
            feedback.append(f"**Question {i+1}:** Wrong. (Your answer: {user_choice} | Correct: {correct_answer})")

    feedback_text = f"### Final Score: {score} / {len(quiz_data)}\n\n" + "\n\n".join(feedback)
    return feedback_text


# --- Gradio UI Layout ---
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Noto+Serif+TC:wght@400;500;600;700&display=swap');

*, *::before, *::after {
    font-family: 'Noto Serif TC', Georgia, serif !important;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
    gr.Markdown("# Mandarin Reading Level Adapter & Quiz Generator")
    gr.Markdown("Enter your text below, select a target difficulty level, and generate a customized reading passage along with a comprehension quiz.")

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(lines=8, label="Original Text", placeholder="Paste the Mandarin text here...")
            level_dropdown = gr.Dropdown(choices=["1", "2", "3", "4", "5", "6", "7"], label="Target Level", value="1")
            generate_btn = gr.Button("Rewrite Text & Generate Quiz", variant="primary")

        with gr.Column():
            output_text = gr.Textbox(lines=8, label="Rewritten Text", interactive=False)

    # Score display
    score_output = gr.Markdown(label="Score", value="")

    # State variable to hold the generated JSON quiz data behind the scenes
    quiz_state = gr.State([])

    # Quiz Section (Initially Hidden)
    quiz_group = gr.Group(visible=False)
    with quiz_group:
        gr.Markdown("### Reading Comprehension Quiz")

        # Pre-allocate 5 radio buttons for the questions
        q1_radio = gr.Radio(label="Q1", choices=[], visible=False)
        q2_radio = gr.Radio(label="Q2", choices=[], visible=False)
        q3_radio = gr.Radio(label="Q3", choices=[], visible=False)
        q4_radio = gr.Radio(label="Q4", choices=[], visible=False)
        q5_radio = gr.Radio(label="Q5", choices=[], visible=False)

        submit_quiz_btn = gr.Button("Submit Answers", variant="primary")
        quiz_results = gr.Markdown(label="Results")

    # --- Event Wiring ---
    radios = [q1_radio, q2_radio, q3_radio, q4_radio, q5_radio]

    generate_btn.click(
        fn=process_text_and_quiz,
        inputs=[input_text, level_dropdown],
        outputs=[output_text, score_output, quiz_state, quiz_group] + radios
    )

    submit_quiz_btn.click(
        fn=grade_quiz,
        inputs=[quiz_state] + radios,
        outputs=quiz_results
    )

if __name__ == "__main__":
    demo.launch(debug=True)


/tmp/ipykernel_692/1369677555.py:98: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:
/tmp/ipykernel_692/1369677555.py:98: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3d559579c2d2e7e907.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 3 rewritting, please wait

 Quiz generating, please wait!
Level 3 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 5 rewritting, please wait

 Quiz generating, please wait!
Level 3 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 3 rewritting, please wait

 Quiz generating, please wait!
Level 6 rewritting, please wait

 Quiz generating, please wait!
Level 1 rewritting, please wait

 Quiz generating, please wait!
Level 3 rewritting, please wait

 Quiz generating, please wait!
Level 5 rewritting, please wait

 Quiz generating, please wait!
Level 4 rewritting, please wait

 Quiz g